###  Successors and Predecessors 

A successor is the next node in a traversal order. 

A predecessor is the previous node in that order. 

So always specify the traversal: inorder successor, preorder predecessor, and so on. These refer to the nodes not necessarily the next larger or the next smaller value.

Consider the below tree:

root = Node(
    4,
    Node(2, Node(1), Node(3)),
    Node(7, Node(6), Node(9))
)

The traversal sequences are ... 

| Traversal | Sequence |
|---|---|
| Inorder | `1, 2, 3, 4, 6, 7, 9` |
| Preorder | `4, 2, 1, 3, 7, 6, 9` |
| Postorder | `1, 3, 2, 6, 9, 7, 4` |
| Level order / BFS | `4, 2, 7, 1, 3, 6, 9` |

Say for node 4,, look immediately before and after i:

| Traversal | Predecessor of `4` | Successor of `4` |
|---|---|---|
| Inorder | `3` | `6` |
| Preorder | None | `2` |
| Postorder | `7` | None |
| Level order | None | `2` |

The first node has no predecessor, the last has no successor.

For the three standard DFS orders, there are six:
- Inorder successor and predecessor.
- Preorder successor and predecessor.
- Postorder successor and predecessor.

Including level order gives eight. These are traversal-relative relationships, rather than eight separate data structures.


### The important distinction: binary tree versus BST

In a general binary tree, inorder successor simply means the next node visited by inorder traversal. Its value could be smaller or larger.

In a BST with distinct keys, inorder traversal is sorted, so:
- Inorder successor: node with the smallest key greater than the current key.
- Inorder predecessor: node with the largest key smaller than the current key.
With duplicate keys, adjacent nodes in inorder may have equal values.

How to find an inorder successor

There are two cases:
| Situation | Successor |
|---|---|
| The node has a right subtree | The **leftmost node in its right subtree** |
| The node has no right subtree | The nearest ancestor for which the node lies in that ancestor’s **left subtree** |

Examples from this tree:
- Successor of 4 is 6: go right to 7, then left to 6.
- Successor of 3 is 4: climb past 2; 4 is the first qualifying ancestor.
- Successor of 9 is None: it is last in inorder.

The predecessor rules are the mirror image:

| Situation | Predecessor |
|---|---|
| The node has a left subtree | The **rightmost node in its left subtree** |
| The node has no left subtree | The nearest ancestor for which the node lies in that ancestor’s **right subtree** |

These structural rules apply to any binary tree. A BST additionally lets you use key comparisons to find ancestors from the root when parent pointers aren’t available.

Where are these useful?

| Relationship | Practical use |
|---|---|
| **Inorder successor/predecessor in a BST** | Moving forward/backward through an ordered collection; retrieving adjacent keys; range iteration |
| **Inorder successor/predecessor during BST deletion** | Replacing a node that has two children while preserving BST ordering |
| **Preorder successor** | Advancing a parent-before-children iterator, such as walking a hierarchy or syntax tree |
| **Postorder successor** | Advancing a children-before-parent iterator, such as processing expression-tree results or cleaning up a hierarchy |
| **Level-order successor** | Advancing through a hierarchy one depth level at a time |


Applications often use an iterator, stack, or queue to maintain this order rather than implement a separately named “successor” function.

For coding interviews, which should you focus on?

**Prioritise inorder successor/predecessor in a BST. They connect directly to several useful problem types:**

| Problem wording | What to recognise |
|---|---|
| “Find the next larger key” | Inorder successor in a BST |
| “Find the previous smaller key” | Inorder predecessor in a BST |
| “Delete a BST node with two children” | Replace using successor or predecessor, then remove that replacement node from its old position |
| “Implement a BST iterator” | Maintain inorder traversal state with a stack |
| “Find the kth smallest element” | Count nodes visited in inorder |
| “Find the next node in preorder” | Use preorder structure; key comparisons aren’t relevant |

Preorder and postorder successor questions are more specialised. For your preparation, first be comfortable with all three traversals, then learn the two inorder-successor cases and their mirrored predecessor cases.

1. Understand the goal through inorder order 

In [ ]:
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value 
        self.left = left 
        self.right = right 

root = Node(
    20, 
    Node(10, Node(5), Node(15)),
    Node(30, Node(25), Node(35))
)

# Its inorder traversal is 
# 5, 10, 15, 20, 25, 30, 35


Therefore:

| Target | Predecessor: next smaller | Successor: next larger |
|---:|---:|---:|
| 5 | None | 10 |
| 15 | 10 | 20 |
| 20 | 15 | 25 |
| 35 | 30 | None |

You could generate the entire inorder list and find these neighbours, but that costs O(n) time and space. BST ordering lets you search along just one path.


2. Inorder successor - Keep the best larger candidate

At each node: 
- If its value is greater than the target, it is a candidate. Go left to look for a smaller qualifying value.
- Otherwise, go right because this node and its left subtree cannot qualify.

In [ ]:
def successor(root, target):
    candidate = None

    while root:
        if root.value > target:
            candidate = root
            root = root.left
        else:
            root = root.right

    return candidate

This returns a node, or None. The argument target is a value.

Worked example: successor of 15

| Current node | Decision | Candidate | Next |
|---:|---|---:|---|
| 20 | Greater than 15; save it | 20 | Left to 10 |
| 10 | Too small | 20 | Right to 15 |
| 15 | Equal; need strictly greater | 20 | Right to None |

Return node 20.

Notice that 15 has no right subtree. We still find its successor because we remembered the qualifying ancestor.

Worked example: successor of 20

| Current node | Decision | Candidate | Next |
|---:|---|---:|---|
| 20 | Equal; need strictly greater | None | Right to 30 |
| 30 | Greater; save it | 30 | Left to 25 |
| 25 | Greater and better; save it | 25 | Left to None |

Return node 25—the leftmost node in 20’s right subtree.

The same implementation handles both structural cases, without separate branches for them.

3. Inorder predecessor — mirror the logic

At each node:
- If its value is less than the target, save it. Go right to look for a larger qualifying value.
- Otherwise, go left.

In [ ]:
def predecessor(root, target):
    candidate = None

    while root:
        if root.value < target:
            candidate = root
            root = root.right
        else:
            root = root.left

    return candidate

Worked example: predecessor of 20

| Current node | Decision | Candidate | Next |
|---:|---|---:|---|
| 20 | Equal; need strictly smaller | None | Left to 10 |
| 10 | Smaller; save it | 10 | Right to 15 |
| 15 | Smaller and better; save it | 15 | Right to None |

Return node 15.

4. Run both functions

Use the tree and functions above:

In [ ]:
for target in [5, 15, 20, 35]:
    prev_node = predecessor(root, target)
    next_node = successor(root, target)

    print(
        f"Target={target}, "
        f"predecessor={prev_node.value if prev_node else None}, "
        f"successor={next_node.value if next_node else None}"
    )

Both functions take O(h) time and O(1) extra space:
- Balanced BST: O(log n) time.
- Skewed BST: O(n) time.
Neither changes the tree.

These functions also work when the target value is absent. For target = 22, they return 20 and 25: the nearest strictly smaller/larger keys. If an interview gives you a target node in a distinct-key BST, you can use its .value.

5. Essential edge cases

| Case | Expected behaviour |
|---|---|
| Empty tree | Both return `None` |
| Single node, target equals its value | Both return `None` |
| Target is minimum | No predecessor |
| Target is maximum | No successor |
| Skewed tree | Correct results; potentially O(n) search |
| Target absent, if allowed | Nearest strictly smaller/larger keys |

6. Exercises — try these without looking at the functions



Exercise 1: Trace both searches

Using the example BST, find the predecessor and successor for:

targets = [10, 25, 30, 22]

For each search, write down the visited nodes and how candidate changes.

Hint: Don’t construct the inorder list until you want to verify your answers.




Exercise 2: Floor and ceiling

Given a BST and a target value, return:
- Floor: largest value less than or equal to the target.
- Ceiling: smallest value greater than or equal to the target.
Return None when an answer doesn’t exist.

Target 22 → floor 20, ceiling 25
Target 20 → floor 20, ceiling 20
Hint: Compare “strictly greater/smaller” with “greater/smaller or equal.” Only a small change is needed.

Exercise 3: Kth smallest value

Given a BST and a valid integer k, return its kth smallest value.

Example tree, k = 3 → 15

Use an iterative solution that stops once the answer is found.

Hint: Your iterative inorder traversal already visits values in the required order.

Exercise 4: BST iterator

Implement:
iterator = BSTIterator(root)
iterator.has_next()  # Whether another value exists
iterator.next()      # Return the next smallest value

Do not store the complete inorder traversal in advance.

Hint: Keep the inorder stack between method calls. Each next() resumes the traversal where the previous call stopped.
